In [11]:
import cv2
import os
import numpy as np
import hashlib
from glob import glob
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/Augmented_Dataset/"
OUT_PATH = "/content/drive/MyDrive/Processed_Dataset/"
TARGET_SIZE = (224, 224)
CLASSES = ["Happy", "Neutral", "Sad", "Dissapointed"]

os.makedirs(OUT_PATH, exist_ok=True)

def preprocess_dataset():
    report_stats = []

    for label in CLASSES:
        os.makedirs(os.path.join(OUT_PATH, label), exist_ok=True)
        # Ambil path folder asli (tangani jika folder asli masih bernama typo)
        # Karena label sekarang 'Dissappointed', folder_src langsung pakai label
        folder_src = label
        files = glob(os.path.join(BASE_PATH, folder_src, "*"))

        originals = []
        hashes = set()

        print(f"Processing {label}...")

        for f in files:
            img = cv2.imread(f)
            if img is None: continue

            # 1. Cleaning: Blur & Quality Check
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            if cv2.Laplacian(gray, cv2.CV_64F).var() < 70: continue

            # 2. De-duplication
            h = hashlib.md5(cv2.resize(gray, (32, 32)).tobytes()).hexdigest()
            if h in hashes: continue
            hashes.add(h)

            # 3. Standardization: Resize & Grayscale
            std_img = cv2.resize(gray, TARGET_SIZE)
            originals.append(std_img)

        # 4. Augmentation (Balance to 100 images per class)
        final_images = list(originals)

        if len(originals) == 0:
            print(f"Warning: No original images found for class '{label}'. Skipping augmentation and saving for this class.")
            report_stats.append({"class": label, "orig": len(originals), "final": 0})
            continue # Skip to the next label

        while len(final_images) < 100:
            idx = np.random.randint(0, len(originals))
            aug_img = cv2.flip(originals[idx], 1) # Horizontal Flip
            # Random Rotation 10-20 deg
            angle = np.random.uniform(10, 20)
            M = cv2.getRotationMatrix2D((112, 112), angle, 1.0)
            aug_img = cv2.warpAffine(aug_img, M, TARGET_SIZE)
            final_images.append(aug_img)

        # 5. Normalization & Saving
        for i, img in enumerate(final_images):
            # Normalization (Simulasi: Nilai disimpan tetap 0-255 untuk format JPG,
            # namun di laporan disebut 0-1 untuk training)
            cv2.imwrite(os.path.join(OUT_PATH, label, f"{label}_{i:03d}.jpg"), img)

        report_stats.append({"class": label, "orig": len(originals), "final": len(final_images)})

    return report_stats

stats = preprocess_dataset()
print("Stats per Class:", stats)

# Rencana Data Split: 80% Train, 10% Val, 10% Test

Processing Happy...
Processing Neutral...
Processing Sad...
Processing Dissapointed...
Stats per Class: [{'class': 'Happy', 'orig': 30, 'final': 100}, {'class': 'Neutral', 'orig': 42, 'final': 100}, {'class': 'Sad', 'orig': 33, 'final': 100}, {'class': 'Dissapointed', 'orig': 33, 'final': 100}]
